# Emotion Insight — v2 (Channel/Size-Aware)


Your model expects **3-channel RGB** input, but the earlier notebook fed **1-channel grayscale**.
This version inspects the model's `input_shape` and automatically adjusts **image size and channels (1 or 3)**.


In [ ]:

from pathlib import Path
from typing import Tuple, Optional
import numpy as np
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

EMOTION_LABELS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
label_to_text = {i: name for i, name in enumerate(EMOTION_LABELS)}

MODEL_PATH: Optional[str] = None  # e.g., '/mnt/data/fer_rgb_model.h5'
DEFAULT_IMG_SIZE: Tuple[int, int] = (48, 48)


In [ ]:

class HeuristicModel:
    """A trivial fallback that works with either 1 or 3 channels."""
    # Expose a Keras-like input_shape for compatibility: (None, H, W, C)
    input_shape = (None, DEFAULT_IMG_SIZE[0], DEFAULT_IMG_SIZE[1], 1)

    def predict(self, batch: np.ndarray) -> np.ndarray:
        N = batch.shape[0]
        preds = np.zeros((N, len(EMOTION_LABELS)), dtype=np.float32)
        # Accept 1ch or 3ch batches
        if batch.ndim != 4:
            raise ValueError("Expected batch with shape (N, H, W, C).")
        ch = batch.shape[-1]
        for i in range(N):
            img = batch[i]
            if ch == 3:
                # convert to gray-ish for heuristic
                img = 0.299*img[...,0] + 0.587*img[...,1] + 0.114*img[...,2]
            else:
                img = img[...,0]
            mean_val = float(img.mean())
            std_val = float(img.std())
            scores = np.zeros(len(EMOTION_LABELS), dtype=np.float32)
            scores[EMOTION_LABELS.index("happy")] = mean_val
            scores[EMOTION_LABELS.index("surprise")] = std_val
            scores[EMOTION_LABELS.index("neutral")] = 0.4
            scores = scores / (scores.sum() if scores.sum() > 0 else len(scores))
            preds[i] = scores
        return preds

def load_model_safely(model_path: Optional[str]):
    if model_path is None:
        print("MODEL_PATH not set — using HeuristicModel fallback.")
        return HeuristicModel()
    try:
        import tensorflow as tf
        model = tf.keras.models.load_model(model_path)
        print(f"Loaded TensorFlow model from: {model_path}")
        return model
    except Exception as e:
        print(f"Failed to load TensorFlow model from '{model_path}' ({e}).\n"
              "Falling back to HeuristicModel.")
        return HeuristicModel()

model = load_model_safely(MODEL_PATH)


In [ ]:

def infer_input_hw_c(model, default_size=(48,48), default_c=1):
    # Try to derive (H, W, C) from model.input_shape
    H, W, C = default_size[0], default_size[1], default_c
    try:
        shape = getattr(model, 'input_shape', None)
        if shape is None and hasattr(model, 'inputs'):
            # For some TF models, use first tensor in .inputs
            shape = getattr(model.inputs[0], 'shape', None)
            if hasattr(shape, 'as_list'):
                shape = shape.as_list()
        if isinstance(shape, (list, tuple)):
            # If the model has multiple inputs, pick the first if nested
            if isinstance(shape[0], (list, tuple)):
                shape = shape[0]
            # Typical Keras: (None, H, W, C)
            if len(shape) >= 4:
                H = int(shape[1]) if shape[1] is not None else H
                W = int(shape[2]) if shape[2] is not None else W
                C = int(shape[3]) if shape[3] is not None else C
    except Exception:
        pass
    return (H, W, C)

def load_image(path: str) -> Image.Image:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Image not found: {p}")
    return Image.open(p)

def preprocess(img: Image.Image, target_size: Tuple[int,int], channels: int) -> np.ndarray:
    img = ImageOps.exif_transpose(img)
    if channels == 1:
        img = img.convert('L')
    else:
        img = img.convert('RGB')
    # Letterbox pad to target_size
    img = ImageOps.pad(img, target_size, method=Image.BILINEAR, color=0, centering=(0.5,0.5))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    if channels == 1:
        arr = arr.reshape(target_size[1], target_size[0],) if arr.ndim == 2 else arr[...,0]
        arr = arr.reshape((1, target_size[1], target_size[0], 1)) if arr.ndim == 2 else arr.reshape((1,)+arr.shape+(1,))
    else:
        # ensure 3 channels
        if arr.ndim == 2:
            arr = np.stack([arr,arr,arr], axis=-1)
        elif arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        arr = arr.reshape((1,) + arr.shape)
    # Keras usually expects (1, H, W, C) where target_size = (W, H) from PIL
    # Ensure axes are (1, H, W, C)
    # PIL returns (H, W, C). So arr is already (1, H, W, C)
    return arr

def predict_emotion(image_path: str):
    img = load_image(image_path)
    H, W, C = infer_input_hw_c(model, default_size=DEFAULT_IMG_SIZE, default_c=1)
    # Note: ImageOps.pad expects size as (W, H)
    batch = preprocess(img, target_size=(W, H), channels=C)
    probs = model.predict(batch)
    if probs.ndim != 2 or probs.shape[1] != len(EMOTION_LABELS):
        if probs.ndim == 1 and probs.size == len(EMOTION_LABELS):
            probs = probs.reshape(1, -1)
        else:
            probs = np.ones((1, len(EMOTION_LABELS)), dtype=np.float32) / len(EMOTION_LABELS)
    idx = int(np.argmax(probs, axis=1)[0])
    return label_to_text[idx], probs[0], (H, W, C)

def show_image_with_label(image_path: str, label: str, probs: np.ndarray):
    img = load_image(image_path)
    # For display only; convert to grayscale to keep it simple
    try:
        disp = img.convert('L')
    except Exception:
        disp = img
    plt.figure()
    plt.imshow(disp, cmap='gray')
    plt.axis('off')
    plt.title(f"Predicted: {label}")
    plt.show()

    plt.figure()
    y = probs
    x = np.arange(len(EMOTION_LABELS))
    plt.bar(x, y)
    plt.xticks(x, EMOTION_LABELS, rotation=30)
    plt.ylabel("Probability")
    plt.title("Emotion probabilities")
    plt.show()


In [ ]:

# Demo: path prompt
test_path = None  # e.g., '/mnt/data/sad_test.jpeg'
path = test_path or input("Enter the path of a face image (or leave blank to skip): ").strip()
if path:
    try:
        label, probs, shape = predict_emotion(path)
        H, W, C = shape
        print(f"Model expects (H,W,C)=({H},{W},{C})")
        print("Emotion detected:", label)
        show_image_with_label(path, probs=probs, label=label)
    except Exception as e:
        print("Failed to run prediction:", e)
else:
    print("Skipped demo. Set `test_path` or enter a valid path when prompted.")
